Training

In [ ]:
import os
import shutil
import json
import requests
import kagglehub

# --- CONFIGURATION ENTRAÎNEMENT ---
BASE_MODEL_ID = "runwayml/stable-diffusion-v1-5"
OUTPUT_DIR = "/tf/workspace/diffusion/simpsons_lora_results"
TRAIN_DATA_DIR = "./local_train_data"
DATASET_NAME = "kostastokis/simpsons-faces"

def setup_environment():
    print("📥 [1/3] Installation des dépendances...")
    # Installation des libs nécessaires pour l'entraînement
    os.system("pip install --root-user-action=ignore -q datasets peft accelerate transformers diffusers bitsandbytes xformers pandas kagglehub")

    # Téléchargement du script officiel d'entraînement HuggingFace
    if not os.path.exists("train_text_to_image_lora.py"):
        print("📥 Téléchargement du script train_text_to_image_lora.py...")
        url = "https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora.py"
        r = requests.get(url)
        with open("train_text_to_image_lora.py", "wb") as f:
            f.write(r.content)
    print("✅ Environnement prêt.")

def prepare_dataset():
    print(f"📂 [2/3] Préparation du dataset '{DATASET_NAME}'...")

    # Téléchargement via KaggleHub
    path = kagglehub.dataset_download(DATASET_NAME)
    source_cropped = os.path.join(path, "cropped") if os.path.exists(os.path.join(path, "cropped")) else path

    # Nettoyage et création du dossier local
    if os.path.exists(TRAIN_DATA_DIR):
        shutil.rmtree(TRAIN_DATA_DIR)
    os.makedirs(TRAIN_DATA_DIR)

    valid_extensions = ('.png', '.jpg', '.jpeg')
    image_files = [f for f in os.listdir(source_cropped) if f.lower().endswith(valid_extensions)]

    metadata_path = os.path.join(TRAIN_DATA_DIR, "metadata.jsonl")

    print(f"   Copie de {len(image_files)} images et création des métadonnées...")
    with open(metadata_path, 'w') as f:
        for img in image_files:
            shutil.copy(os.path.join(source_cropped, img), os.path.join(TRAIN_DATA_DIR, img))
            # Prompt d'entraînement avec le mot clé déclencheur "sks"
            line = {"file_name": img, "text": "face of sks simpson character"}
            f.write(json.dumps(line) + "\n")

    print(f"✅ Dataset prêt dans : {TRAIN_DATA_DIR}")

def start_training():
    print("🚀 [3/3] Lancement de l'entraînement LoRA...")

    # Commande accelerate optimisée pour T4/P100 (Google Colab/Kaggle)
    cmd = f"""
    accelerate launch train_text_to_image_lora.py \
      --pretrained_model_name_or_path="{BASE_MODEL_ID}" \
      --train_data_dir="{TRAIN_DATA_DIR}" \
      --dataset_name="imagefolder" \
      --resolution=512 \
      --center_crop \
      --random_flip \
      --train_batch_size=2 \
      --gradient_accumulation_steps=2 \
      --max_train_steps=1000 \
      --learning_rate=1e-04 \
      --lr_scheduler="cosine" \
      --lr_warmup_steps=100 \
      --output_dir="{OUTPUT_DIR}" \
      --mixed_precision="fp16" \
      --seed=42
    """

    exit_code = os.system(cmd)

    if exit_code == 0:
        print(f"🎉 Entraînement terminé avec succès ! Le modèle est dans : {OUTPUT_DIR}")
        print("👉 Vous pouvez maintenant lancer le fichier 'generate.py'")
    else:
        print(f"❌ Une erreur est survenue (Code {exit_code})")

if __name__ == "__main__":
    setup_environment()
    prepare_dataset()
    start_training()

Generator

In [ ]:
import os
from datetime import datetime
import torch
from diffusers import StableDiffusionPipeline

# --- CONFIGURATION GÉNÉRATION ---
# Chemin où train.py a sauvegardé le modèle
LORA_PATH = "/tf/workspace/diffusion/simpsons_lora_results/pytorch_lora_weights.safetensors"

# Modèle RÉALISTE pour l'inférence (Le secret du rendu photoréaliste)
REALISTIC_MODEL_ID = "SG161222/Realistic_Vision_V6.0_B1_noVAE"
# Alternative si le premier est trop lourd : "runwayml/stable-diffusion-v1-5"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_pipeline():
    print(f"🧠 Chargement du modèle de base : {REALISTIC_MODEL_ID}...")
    try:
        pipe = StableDiffusionPipeline.from_pretrained(
            REALISTIC_MODEL_ID,
            torch_dtype=torch.float16,
            safety_checker=None,
            requires_safety_checker=False
        ).to(DEVICE)
    except Exception as e:
        print(f"⚠️ Erreur chargement modèle réaliste ({e}). Vérifiez votre connexion.")
        return None

    if not os.path.exists(LORA_PATH):
        print(f"❌ ERREUR : Le fichier LoRA est introuvable ici : {LORA_PATH}")
        print("   Avez-vous bien exécuté 'train.py' avant ?")
        return None

    print(f"🔌 Injection des poids LoRA Simpsons...")
    try:
        pipe.load_lora_weights(os.path.dirname(LORA_PATH), weight_name="pytorch_lora_weights.safetensors")
    except Exception as e:
        print(f"❌ Erreur lors du chargement du LoRA : {e}")
        return None

    return pipe

def generate(pipe):
    while True:
        subject = input("\n👤 Quel personnage voulez-vous générer ? (ex: Homer Simpson) [ou 'q' pour quitter] : ")
        if subject.lower() == 'q':
            break

        # Prompt conçu pour forcer le réalisme et supprimer le style cartoon
        prompt = (
            f"raw photo of {subject}, sks simpson style, "
            "hyper realistic, 8k, detailed skin texture, pore details, "
            "cinematic lighting, shallow depth of field, sharp focus, "
            "photograph taken on Canon 5D, masterpiece"
        )

        negative_prompt = (
            "cartoon, drawing, anime, 2d, illustration, painting, yellow skin, "
            "flat colors, cel shading, low quality, blurry, deformed, distorted, "
            "bad anatomy, ugly"
        )

        print(f"🎨 Génération en cours pour '{subject}'...")

        # Le paramètre cross_attention_kwargs={"scale": ...} est CRUCIAL.
        # 0.6 est un bon équilibre : assez de traits Simpsons, mais texture humaine.
        scale = 0.6

        image = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=30,
            guidance_scale=6.0,
            cross_attention_kwargs={"scale": scale}
        ).images[0]

        short_name = subject.split()[0].lower()
        timestamp = datetime.now().strftime("%H%M%S")

        filename = f"/tf/workspace/diffusion/images/{short_name}_{timestamp}.png"
        image.save(filename)
        print(f"✨ Image sauvegardée : {filename}")

if __name__ == "__main__":
    if DEVICE == "cpu":
        print("⚠️ ATTENTION : Vous tournez sur CPU. La génération sera très lente.")

    pipeline = load_pipeline()
    if pipeline:
        generate(pipeline)

In [ ]:
import os
import torch
from diffusers import StableDiffusionPipeline, AutoencoderKL
from datetime import datetime

# --- CONFIGURATION ---
LORA_PATH = "simpsons_lora_results/pytorch_lora_weights.safetensors"
# On garde le modèle réaliste
REALISTIC_MODEL_ID = "SG161222/Realistic_Vision_V6.0_B1_noVAE"
# On ajoute un VAE spécifique pour corriger les visages et les taches noires
VAE_ID = "stabilityai/sd-vae-ft-mse"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_FOLDER = "/tf/workspace/diffusion/outputs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def load_pipeline():
    print(f"🔧 Chargement du VAE correctif (pour éviter les taches noires)...")
    try:
        # On charge le VAE séparément
        vae = AutoencoderKL.from_pretrained(VAE_ID, torch_dtype=torch.float16).to(DEVICE)
    except Exception as e:
        print(f"⚠️ Impossible de charger le VAE : {e}")
        vae = None

    print(f"🧠 Chargement du modèle réaliste...")
    try:
        pipe = StableDiffusionPipeline.from_pretrained(
            REALISTIC_MODEL_ID,
            torch_dtype=torch.float16,
            vae=vae, # On injecte le VAE ici
            safety_checker=None,
            requires_safety_checker=False
        ).to(DEVICE)
    except Exception as e:
        print(f"⚠️ Erreur chargement modèle ({e}).")
        return None

    print(f"🔌 Chargement du LoRA Simpsons...")
    try:
        if os.path.exists(LORA_PATH):
            pipe.load_lora_weights(os.path.dirname(LORA_PATH), weight_name="pytorch_lora_weights.safetensors")
        else:
            print(f"❌ FICHIER LORA NON TROUVÉ : {LORA_PATH}")
            return None
    except Exception as e:
        print(f"❌ Erreur LoRA : {e}")
        return None

    return pipe

def generate(pipe):
    while True:
        subject = input("\n👤 Personnage (ex: Homer) [ou 'q' pour quitter] : ")
        if subject.lower() == 'q':
            break

        # Prompt encore plus renforcé pour le réalisme
        prompt = (
            f"raw photo of {subject}, sks simpson style, "
            "detailed skin texture, pore details, hyper realistic, "
            "soft cinematic lighting, photorealistic, 8k uhd, "
            "dslr, sharp focus, high quality"
        )

        # Negative prompt pour empêcher le dessin animé
        negative_prompt = (
            "cartoon, anime, 3d render, painting, drawing, illustration, "
            "disfigured, bad anatomy, deformed face, black spots, noise, "
            "grainy, low resolution, blurry"
        )

        print(f"🎨 Génération de '{subject}'...")

        image = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=30,
            guidance_scale=5.0, # Un peu plus bas pour éviter de 'brûler' l'image
            cross_attention_kwargs={"scale": 0.6}, # Force du LoRA
            height=768, # Hauteur augmentée pour de meilleurs visages
            width=512
        ).images[0]

        short_name = subject.split()[0].lower()
        timestamp = datetime.now().strftime("%H%M%S")
        filename = f"{short_name}_{timestamp}.png"
        save_path = os.path.join(OUTPUT_FOLDER, filename)

        image.save(save_path)
        print(f"✨ Sauvegardé : {save_path}")

if __name__ == "__main__":
    pipeline = load_pipeline()
    if pipeline:
        generate(pipeline)

In [ ]:
import os
import glob
from IPython.display import display, Image

# Dossier où sont les images (le dossier courant par défaut)
image_folder = "/tf/workspace/diffusion/images"

# Récupère tous les fichiers PNG
images = glob.glob(os.path.join(image_folder, "*.png"))

if not images:
    print("❌ Aucune image trouvée dans ce dossier.")
else:
    print(f"👀 Affichage de {len(images)} images générées :\n")
    for img_path in images:
        print(f"📂 Fichier : {img_path}")
        # Affiche l'image avec une largeur max pour ne pas exploser l'écran
        display(Image(filename=img_path, width=512))